In [1]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.spatial.distance import cdist
import json

# Load best config from Module 5
with open("best_config.json", "r") as f:
    best_config = json.load(f)

BEST_MODEL  = best_config["model_name"]
BEST_METRIC = best_config["metric"]

print("Best model  :", BEST_MODEL)
print("Best metric :", BEST_METRIC)

c:\Users\Hemanth Sai\OneDrive\Desktop\info\queytube\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Best model  : all-MiniLM-L6-v2
Best metric : cosine


In [2]:
# Load video index
index_df = pd.read_csv("video_index.csv")

# Extract embedding columns
embedding_cols   = [c for c in index_df.columns if c.startswith("embedding_")]
doc_embeddings   = index_df[embedding_cols].values  # shape: (n_videos, emb_dim)
video_ids        = index_df["video_id"].tolist()
titles           = index_df["title"].tolist()

print(f"Videos in index  : {len(video_ids)}")
print(f"Embedding dim    : {doc_embeddings.shape[1]}")

# Load model
print(f"\nLoading model: {BEST_MODEL}")
model = SentenceTransformer(BEST_MODEL)
print("Model loaded ✅")

Videos in index  : 244
Embedding dim    : 384

Loading model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2051.32it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded ✅


In [3]:
def encode_query(query: str) -> np.ndarray:
    """Convert a text query into an embedding vector."""
    return model.encode([query], convert_to_numpy=True)

In [4]:
def compute_cosine(query_emb: np.ndarray) -> np.ndarray:
    return cosine_similarity(query_emb, doc_embeddings)[0]  # higher = better

# Quick test
test_emb = encode_query("learn Python")
cos_scores = compute_cosine(test_emb)
print("Cosine score range:", cos_scores.min().round(4), "to", cos_scores.max().round(4))

Cosine score range: -0.0514 to 0.6143


In [5]:
def compute_euclidean(query_emb: np.ndarray) -> np.ndarray:
    return -cdist(query_emb, doc_embeddings, metric="euclidean")[0]  # negate: higher = closer

def compute_manhattan(query_emb: np.ndarray) -> np.ndarray:
    return -cdist(query_emb, doc_embeddings, metric="cityblock")[0]

euclid_scores = compute_euclidean(test_emb)
manhat_scores = compute_manhattan(test_emb)
print("Euclidean score range:", euclid_scores.min().round(4), "to", euclid_scores.max().round(4))
print("Manhattan score range:", manhat_scores.min().round(4), "to", manhat_scores.max().round(4))

Euclidean score range: -1.4024 to -0.8423
Manhattan score range: -21.8404 to -13.1331


In [8]:
# Regenerate separate title/transcript embeddings for weighted fusion
# (Only needed if you want to test different title:transcript weight ratios)

clean_df = pd.read_csv("cleaned_transcripts.csv")
clean_df["title"]      = clean_df["title"].fillna("")
clean_df["transcript"] = clean_df["transcript"].fillna("")

print("Encoding title embeddings for weighted fusion...")
title_embs = model.encode(clean_df["title"].tolist(),      convert_to_numpy=True, show_progress_bar=True)

print("Encoding transcript embeddings for weighted fusion...")
trans_embs = model.encode(clean_df["transcript"].tolist(), convert_to_numpy=True, show_progress_bar=True)

print("Done. Shapes:", title_embs.shape, trans_embs.shape)

Encoding title embeddings for weighted fusion...


Batches: 100%|██████████| 8/8 [00:01<00:00,  5.79it/s]


Encoding transcript embeddings for weighted fusion...


Batches: 100%|██████████| 8/8 [00:32<00:00,  4.10s/it]

Done. Shapes: (244, 384) (244, 384)


In [ ]:
def compute_combined_score(query_emb: np.ndarray,
                           title_weight: float = 0.3,
                           transcript_weight: float = 0.7,
                           metric: str = "cosine") -> np.ndarray:
    """
    Compute a weighted fusion of title and transcript scores.
    Default: 30% title, 70% transcript (transcripts carry more content).
    """
    if metric == "cosine":
        t_scores  = cosine_similarity(query_emb, title_embs)[0]
        tr_scores = cosine_similarity(query_emb, trans_embs)[0]
    
    else:
        raise ValueError(f"Unknown metric: {metric}")

    return title_weight * t_scores + transcript_weight * tr_scores

# Quick test
test_emb = encode_query("explain neural networks")
combined_scores = compute_combined_score(test_emb, metric="cosine")
print("Combined score range:", combined_scores.min().round(4), "to", combined_scores.max().round(4))

Combined score range: -0.0813 to 0.4393


In [10]:
def rank_results(scores: np.ndarray, top_k: int = 5, threshold=None) -> list:
    """
    Sort videos by score (descending), apply optional threshold, return top_k.
    """
    ranked_idx = np.argsort(scores)[::-1]
    results = []

    for idx in ranked_idx:
        if threshold is not None and scores[idx] < threshold:
            continue
        results.append({
            "video_id" : video_ids[idx],
            "title"    : titles[idx],
            "score"    : round(float(scores[idx]), 4)
        })
        if len(results) == top_k:
            break

    return results

In [11]:
# Demonstrate threshold effect on a sample query
test_query = "how to build a REST API with Python"
test_emb   = encode_query(test_query)
scores     = compute_combined_score(test_emb, metric=BEST_METRIC)

print(f"Query: '{test_query}'")
print()

# No threshold
print("--- No threshold (top 5) ---")
for r in rank_results(scores, top_k=5):
    print(f"  score={r['score']:7.4f} | {r['title'][:65]}")

print()

# Try different thresholds and see how many results survive
cosine_thresholds  = [0.3, 0.4, 0.5, 0.6]
distance_thresholds = [-1.5, -1.2, -1.0, -0.8]

thresholds_to_try = cosine_thresholds if BEST_METRIC == "cosine" else distance_thresholds

print(f"--- Threshold sweep (metric={BEST_METRIC}) ---")
for thresh in thresholds_to_try:
    results = rank_results(scores, top_k=10, threshold=thresh)
    print(f"  threshold={thresh:5.2f}  →  {len(results)} results returned")

Query: 'how to build a REST API with Python'

--- No threshold (top 5) ---
  score= 0.3918 | Python Essentials for AI Agents Tutorial
  score= 0.2934 | Cryptography for Beginners Full Python Course SHA256 AES RSA Pass
  score= 0.2931 | Intro to Backend Web Development Nodejs Express Tutorial for Begi
  score= 0.2626 | How to use the title method in Python
  score= 0.2583 | Have you ever wondered where Python got its name

--- Threshold sweep (metric=cosine) ---
  threshold= 0.30  →  1 results returned
  threshold= 0.40  →  0 results returned
  threshold= 0.50  →  0 results returned
  threshold= 0.60  →  0 results returned


In [24]:
def returnSearchResults(query: str,
                        df: pd.DataFrame = index_df,
                        metric: str = BEST_METRIC,
                        title_weight: float = 0.3,
                        transcript_weight: float = 0.7,
                        top_k: int = 5,
                        threshold=None) -> list:
    # Step 1: Encode
    query_emb = encode_query(query)

    # Step 2: Score
    scores = compute_combined_score(
        query_emb,
        title_weight=title_weight,
        transcript_weight=transcript_weight,
        metric=metric
    )

    # Step 3 & 4: Rank + threshold
    raw_results = rank_results(scores, top_k=top_k, threshold=0.3)

    # Add YouTube links
    for r in raw_results:
        r["youtube_link"] = f"https://www.youtube.com/watch?v={r['video_id']}"

    return raw_results


# ---- Test it ----
results = returnSearchResults("what are LLMs and how do they work")

print("Query: 'what are LLMs and how do they work'")
print("Top 5 Results:")
print("-" * 65)
for i, r in enumerate(results, 1):
    print(f"Rank {i}: {r['title'][:60]}")
    print(f"         Link  : {r['youtube_link']}")
    print(f"         Score : {r['score']}")
    print()

Query: 'what are LLMs and how do they work'
Top 5 Results:
-----------------------------------------------------------------
Rank 1: Learn the basics of LLMs in 60 seconds with Beau Carnes
         Link  : https://www.youtube.com/watch?v=VWcCVKaN2vA
         Score : 0.594

Rank 2: LLMs havent really gotten smarter but the tools we use with 
         Link  : https://www.youtube.com/watch?v=H_3sWUR9v7A
         Score : 0.404

Rank 3: LLM FineTuning Course From Supervised FT to RLHF LoRA and Mu
         Link  : https://www.youtube.com/watch?v=CcrC5zSv1iA
         Score : 0.3332

Rank 4: Learn MLOps with MLflow and Databricks Full Course for Machi
         Link  : https://www.youtube.com/watch?v=tVskbekONlw
         Score : 0.306

